In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.sql import functions as F

from sklearn.metrics import mean_absolute_percentage_error
from xgboost.spark import SparkXGBRegressor

In [0]:
sql_query = """SELECT  cy.Year, cy.Country, cy.Item, cy.ItemCode, cy.Element, cy.Unit, cy.Yield, avg_temp.Temperature, corr_index.CPI as CorruptionIndex, sog.GDP, up.UrbanPopulation, up.RuralPopulation, fu.FertilizerUse,
peia.PeopleEmployedInAgriculture, fw.FreshwaterWithdrawal
FROM agriculture_db.crop_yield cy
INNER JOIN agriculture_db.country_codes cc ON cc.ISONum = cy.AreaCodeM49
INNER JOIN 
(SELECT AVG(ast.Temperature) as Temperature, ast.Year, ast.ISO3
FROM agriculture_db.average_surface_temperature ast
GROUP BY ast.Country, ast.Year, ast.ISO3) avg_temp ON avg_temp.ISO3 = cc.ISO3 AND avg_temp.Year = cy.Year
INNER JOIN agriculture_db.corruption_index corr_index ON corr_index.ISO = cc.ISO3 AND corr_index.Year = cy.Year
INNER JOIN agriculture_db.share_of_gdp sog ON sog.Code = cc.ISO3 AND sog.Year = cy.Year
INNER JOIN agriculture_db.urban_population up ON up.Code = cc.ISO3 AND up.Year = cy.Year
INNER JOIN agriculture_db.fertilizer_use fu ON fu.ISO3 = cc.ISO3 AND fu.Year = cy.Year
INNER JOIN agriculture_db.people_employed_in_agriculture peia ON peia.ISO3 = cc.ISO3 AND peia.Year = cy.Year
INNER JOIN agriculture_db.freshwater_withdrawal fw ON fw.ISO3 = cc.ISO3 AND fw.Year = cy.Year"""
features_df = spark.sql(sql_query)
crop_categories_df = spark.table("agriculture_db.crop_categories")

In [0]:
features_df = features_df.dropna(subset=["Yield", "CorruptionIndex", "FertilizerUse", "FreshwaterWithdrawal"])
features_df = (features_df
    .withColumn("Temperature", F.round(F.col("Temperature"), 2))
)
features_df_category = features_df.join(crop_categories_df, on=["ItemCode", "Item"], how="left")
crops_df = features_df_category.filter(((F.col("Category") == "Vegetables Primary") | 
                                        (F.col("Category") == "Crops, primary") |
                                        (F.col("Category") == "Oilcrops, Oil Equivalent") |
                                        (F.col("Category") == "Citrus Fruit, Total") |
                                        (F.col("Category") == "Cereals, primary") |
                                        (F.col("Category") == "Oilcrops, Cake Equivalent") |
                                        (F.col("Category") == "Sugar Crops Primary") |
                                        (F.col("Category") == "Fruit Primary") |
                                        (F.col("Category") == "Oilcrops Primary") |
                                        (F.col("Category") == "Fibre Crops Primary") |
                                        (F.col("Category") == "Fibre Crops, Fibre Equivalent") |
                                        (F.col("Category") == "Roots and Tubers, Total"))
                                       & (F.col("Element") == "Yield")
                                       & (F.col("Yield") > 0))
vegetables_df = features_df_category.filter(((F.col("Category") == "Vegetables Primary") |
                                             (F.col("Category") == "Roots and Tubers, Total")) 
                                            & (F.col("Element") == "Yield")
                                            & (F.col("Yield") > 0))

In [0]:
vegetables = vegetables_df.groupBy("Item").agg(F.countDistinct("Country").alias("CountryCount")) \
    .where((F.col("CountryCount") > 70) & ~(F.col("Item").rlike("Other|Primary")))

crops = crops_df.groupBy("Item").agg(F.countDistinct("Country").alias("CountryCount")) \
    .where((F.col("CountryCount") > 90) & ~(F.col("Item").rlike("Other|Primary")))

countries = vegetables_df.groupBy("Country").agg(F.countDistinct("Item").alias("ItemCount")) \
    .where(F.col("ItemCount") > 19)

crop_countries = crops_df.groupBy("Country").agg(F.countDistinct("Item").alias("ItemCount")) \
    .where(F.col("ItemCount") > 80)

In [0]:
vegetables_filtered_df = vegetables_df.join(vegetables.select("Item"), "Item", how="inner").join(countries.select("Country"), "Country", how="inner")
crops_filtered_df = crops_df.join(crops.select("Item"), "Item", how="inner").join(crop_countries.select("Country"), "Country", how="inner")

In [0]:
item_indexer = StringIndexer(inputCol="Item", outputCol="ItemIndexed", handleInvalid="keep")
item_encoder = OneHotEncoder(inputCol="ItemIndexed", outputCol="ItemEncoded", dropLast=False)
assembler = VectorAssembler(inputCols=["ItemEncoded", "Temperature", "CorruptionIndex", "GDP", "UrbanPopulation", "RuralPopulation", "FertilizerUse", "FreshwaterWithdrawal"], outputCol="features")

xgboost_model = SparkXGBRegressor(features_col="features", label_col="Yield", max_depth=7, learning_rate=0.2, min_split_loss=1.0, n_estimators=200, subsample=0.5, colsample_bytree=0.8)
xgboost_pipeline = Pipeline(stages=[item_indexer, item_encoder, assembler, xgboost_model])

In [0]:
train_data, test_data = vegetables_filtered_df.randomSplit([0.8, 0.2], seed=42)
xgboost_fit_model = xgboost_pipeline.fit(train_data)
predictions = xgboost_fit_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
true = [row["Yield"] for row in predictions.select("Yield").collect()]
pred = [row["prediction"] for row in predictions.select("prediction").collect()]
mean_absolute_percentage_error(true, pred)

In [0]:
df_pd = predictions.select("Yield", "prediction").toPandas()

plt.figure(figsize=(8, 6))
sns.scatterplot(x=df_pd["Yield"], y=df_pd["prediction"])

plt.xlabel("True")
plt.ylabel("Pred")
plt.axline((0, 0), slope=1, color="black", label="Perfect Prediction")
plt.legend()
plt.show()

In [0]:
train_data, test_data = crops_filtered_df.randomSplit([0.8, 0.2], seed=42)
xgboost_fit_model = xgboost_pipeline.fit(train_data)
predictions = xgboost_fit_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="Yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² (Coefficient of Determination): {r2:.2f}")

In [0]:
true = [row["Yield"] for row in predictions.select("Yield").collect()]
pred = [row["prediction"] for row in predictions.select("prediction").collect()]
print(mean_absolute_percentage_error(true, pred))

In [0]:
df_pd = predictions.select("Yield", "prediction").toPandas()

plt.figure(figsize=(8, 6))
sns.scatterplot(x=df_pd["Yield"], y=df_pd["prediction"])

plt.xlabel("True")
plt.ylabel("Pred")
plt.axline((0, 0), slope=1, color="black", label="Perfect Prediction")
plt.legend()
plt.show()